# tensor-unbind — ex6: batched ray cast with per-step shape debug

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `tensor-unbind`. Running the final beacon cell reports progress against the `Numpy: Indexing and selection` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import matplotlib.pyplot as plt

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Numpy: Indexing and selection` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`tensor-unbind`** (exercise 6). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "tensor-unbind"
DD_SUBTOPIC = "Numpy: Indexing and selection"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## torch unbind — quick refresher

`x.unbind(dim=k)` returns a tuple of `x.shape[k]` view-tensors with axis `k` removed. The result is a *Python tuple*, not a tensor — perfect for destructuring named components (`origin, direction = rays.unbind(dim=1)`) or for fanning a batched tensor into per-head / per-channel slices.

**Compared to `select`.** `unbind(dim=k)[i]` ≡ `select(k, i)`. Use `select` when you want ONE slice; use `unbind` when you want ALL of them. Both return views (no copy), so writes through the view alias the source.

### Exercise 6 — batched ray cast with per-step shape debug

> ```yaml
> Difficulty: 🔴🔴🔴🔴⚪
> Bloom level: Create
> LO: Apply two levels of unbind (rays → origin/direction → x/y/z) to solve the analytic ray-plane intersection for a whole batch, printing shapes at each step.
> Keywords: ray-tracing, shape-debug, two-level-unbind, ground-plane
> ```

**KCs targeted:** `unbind-explicit-dim`, `unbind-tuple-destructure`, `unbind-ray-decomposition`

Implement `ex6_ray_ground_intersect(rays)`. A full batched ray cast against the ground plane `y = 0`:

1. `rays` has shape `(B, 2, 3)` — row 0 of each `(2,3)` block is the origin, row 1 is the direction.
2. First-level unbind: `origin, direction = rays.unbind(dim=1)`. **Print `origin.shape` and `direction.shape`** with descriptive labels so the caller sees the decomposition.
3. Second-level unbind on `origin` and `direction` along the last axis to get `ox, oy, oz` and `dx, dy, dz` (each `(B,)`). **Print `oy.shape` and `dy.shape`**.
4. Solve `origin.y + t * direction.y == 0` for `t`:
   `t_hit = -oy / dy`. Watch for `dy == 0` (ray parallel to ground → produces `inf` or `nan`, which the test tolerates).
5. Compute the hit point with the parametric ray equation, return an `(B, 3)` tensor.

Output: `(B, 3)` `float32` hit points. For parallel rays the row contains `inf` or `nan` (don't try to mask them).

The visualization cell projects the X/Z components onto a 2-D ground-plane scatter so you can see where each ray landed.

In [ ]:
def ex6_ray_ground_intersect(rays: Tensor) -> Tensor:
    """Intersect a (B, 2, 3) batch of rays with the y=0 plane."""
    raise NotImplementedError()


def _test_ex6():
    rays = t.tensor([
        # ray 0 — from (0, 2, 0) pointing straight down → hits (0, 0, 0) at t=2
        [[0.0, 2.0, 0.0], [0.0, -1.0, 0.0]],
        # ray 1 — from (1, 4, 1) pointing down-and-forward → hits (1, 0, 5) at t=4
        [[1.0, 4.0, 1.0], [0.0, -1.0, 1.0]],
        # ray 2 — from (-3, 3, 2) pointing down → hits (-3, 0, 2) at t=3
        [[-3.0, 3.0, 2.0], [0.0, -1.0, 0.0]],
    ])
    hits = ex6_ray_ground_intersect(rays)
    assert hits.shape == (3, 3), f'expected (3,3), got {tuple(hits.shape)}'
    assert hits.dtype == t.float32, f'expected float32, got {hits.dtype}'
    expected = t.tensor([
        [0.0,  0.0, 0.0],
        [1.0,  0.0, 5.0],
        [-3.0, 0.0, 2.0],
    ])
    assert t.allclose(hits, expected, atol=1e-5), f'value mismatch:\n{hits}\nvs\n{expected}'
    # y-component of every hit must be 0 (we hit the ground plane).
    assert t.allclose(hits[:, 1], t.zeros(3), atol=1e-5), 'all hits must have y == 0'

    # Edge case — parallel ray (dy == 0) should produce inf / nan without erroring.
    parallel = t.tensor([[[0.0, 1.0, 0.0], [1.0, 0.0, 0.0]]])  # ray traveling along +x
    p_hit = ex6_ray_ground_intersect(parallel)
    assert p_hit.shape == (1, 3)
    assert not t.isfinite(p_hit).all().item(), 'parallel ray should yield inf/nan, got finite'

    # --- Visualization: scatter hits on the ground plane ---
    rng = t.Generator().manual_seed(11)
    B = 100
    origins = t.stack([
        t.linspace(-5, 5, B),
        t.full((B,), 4.0),
        t.linspace(-3, 3, B),
    ], dim=1)
    directions = t.stack([
        0.3 * t.randn(B, generator=rng),
        t.full((B,), -1.0),
        0.3 * t.randn(B, generator=rng),
    ], dim=1)
    big_rays = t.stack([origins, directions], dim=1)  # (B, 2, 3)
    big_hits = ex6_ray_ground_intersect(big_rays)
    fig, ax = plt.subplots(figsize=(5, 5))
    ax.scatter(big_hits[:, 0].numpy(), big_hits[:, 2].numpy(),
               c=range(B), cmap='viridis', s=20)
    ax.set_xlabel('hit X')
    ax.set_ylabel('hit Z')
    ax.set_title(f'ex6 ground-plane hits (B={B} rays from y=4 downward)')
    ax.set_aspect('equal')
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    _dd_passed.add('ex6')
    print("ex6 ✓")

_test_ex6()

<details><summary>Solution</summary>

```python
def ex6_ray_ground_intersect(rays: Tensor) -> Tensor:
    origin, direction = rays.unbind(dim=1)
    print(f'  origin.shape={tuple(origin.shape)}  direction.shape={tuple(direction.shape)}')
    ox, oy, oz = origin.unbind(dim=-1)
    dx, dy, dz = direction.unbind(dim=-1)
    print(f'  oy.shape={tuple(oy.shape)}  dy.shape={tuple(dy.shape)}')
    t_hit = -oy / dy
    return origin + t_hit.unsqueeze(-1) * direction
```

**Two-level unbind.** The outer `unbind(dim=1)` peels `rays` `(B,2,3)` into two `(B,3)` named tensors. The inner `unbind(dim=-1)` peels each into three `(B,)` scalars, ready for elementwise arithmetic. This is dramatically clearer than `rays[:, 1, 1]` for the y-component of direction.

**Why broadcast with `unsqueeze`.** `t_hit` is `(B,)`; `direction` is `(B,3)`. To multiply them elementwise we need `(B,1) * (B,3)` so broadcast lines up. `t_hit.unsqueeze(-1)` adds the trailing size-1 axis.

**Parallel rays produce `inf`/`nan` — and that's fine.** A real renderer masks them out with `t.isfinite(t_hit)`. The test only asserts the divergence happens; downstream code is responsible for filtering.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex6'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex6',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()